<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-segmentation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>Semantic Segmentation (BDD100k)</b> notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> traces training signals back to the exact samples producing them. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Semantic Segmentation (BDD100k) with WeightsLab

Trains a small U-Net on a **BDD100k** subset and instruments per-**instance** and per-**sample** Dice (metric) and BCE (loss), so every mask's quality is traced back to the exact sample and annotation producing it.

### What you'll do
1. Install WeightsLab.
2. Download your dataset **zip from a Google Drive link** and unzip it (the training **code is inlined** below).
3. Set every knob in one **config** dict (like a `config.yaml`).
4. Wrap the dataloaders, model, optimizer, and per-instance / per-sample Dice + BCE signals with the SDK.
5. Train while per-sample and per-annotation signals are captured, then (optionally) open the live **Weights Studio** UI.

> Data note: this notebook fetches your dataset from a **Drive share link** you provide (`DATASET_URL`); all training code (dataset, model, criterions, loops) is **inlined below** - no `main.py`, no full-repo clone. The zip should unpack to a BDD100k-style layout (`images/{train,val}/` + `labels/{train,val}/`).

## Setup

On Colab, pick a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install -q weightslab
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev3"

### Fetch the dataset from Google Drive

Set **`DATASET_URL`** below to your Drive **share link** for the dataset **zip** (the file must be shared as *Anyone with the link*). The cell downloads it with [`gdown`](https://github.com/wkentaro/gdown), extracts it, and auto-detects the dataset root — the folder that contains an `images/` subdirectory (so it works even if your zip wraps everything in a top-level folder). `config["data_root"]` then points at that folder.

The zip should unpack to the BDD100k layout the dataset class expects:

```
<root>/
  images/{train,val}/   # .jpg / .png inputs
  labels/{train,val}/   # matching .png masks (same basename as the image)
```

In [ ]:
# ===== Dataset DATA from your Google Drive — the training CODE is inlined below =====
# Paste your Drive SHARE link for the dataset zip here (shared as "Anyone with the link").
DATASET_URL = "https://drive.google.com/file/d/1AbCdEfG.../view?usp=sharing"  # <-- replace with your link

import os, sys, subprocess, zipfile

# gdown handles Drive downloads (incl. the large-file virus-scan confirmation).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

zip_path = "dataset.zip"
extract_dir = "dataset"

# Download (fuzzy=True lets gdown accept a full "file/d/<id>/view" share URL).
if not os.path.exists(zip_path):
    gdown.download(url=DATASET_URL, output=zip_path, quiet=False, fuzzy=True)
else:
    print(f"{zip_path} already present — skipping download.")

# Extract.
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)


def _find_data_root(base):
    """Return the folder that holds an `images/` subdir (handles a wrapping top-level dir)."""
    if os.path.isdir(os.path.join(base, "images")):
        return base
    for d, subdirs, _ in os.walk(base):
        if "images" in subdirs:
            return d
    return base


DATA_ROOT = _find_data_root(extract_dir)

print("Dataset root:", DATA_ROOT)
print("Contents:", sorted(os.listdir(DATA_ROOT)))
for _split in ("train", "val"):
    _p = os.path.join(DATA_ROOT, "images", _split)
    if os.path.isdir(_p):
        print(f"  images/{_split}: {len(os.listdir(_p))} files")
    else:
        print(f"  images/{_split}: (missing — check your zip's layout)")

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase. These are the external imports gathered from the example's `main.py` and `utils/` modules.

In [ ]:
import os
import time
import logging
import tempfile

import numpy as np
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

# Setup loggers (from main.py)
logging.basicConfig(level=logging.ERROR)
logging.getLogger("PIL").setLevel(logging.INFO)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. The dataset (inlined `utils/data.py`)

`BDD100kSegDataset` returns `(image, uid, instances, metadata)` per sample, where `instances` is a **list of per-instance masks**. `seg_collate` keeps those variable-length lists intact so WeightsLab can attribute signals per annotation.

In [ ]:
# ===== utils/data.py =====
import os
import torch
import numpy as np

from torchvision import transforms

from torch.utils.data import Dataset

from PIL import Image


# =============================================================================
# BDD100k segmentation dataset
# =============================================================================
class BDD100kSegDataset(Dataset):
    """
    Uses your existing layout:

      data/BDD100k_reduced/
        images_1280x720/
          train/
          val/
        bdd100k_labels_dac_daa_lls_lld_curbs/
          train/
          val/

    Assumes image & label share basename (e.g. 0001.jpg / 0001.png).
    """

    def __init__(
        self,
        root,
        split="train",
        num_classes=6,
        class_names=None,
        ignore_index=255,
        image_size=256,
        max_samples=None
    ):
        super().__init__()
        self.root = root
        self.split = split
        self.num_classes = num_classes
        self.class_names = class_names
        self.ignore_index = ignore_index
        self.task_type = "segmentation"

        img_dir = os.path.join(root, "images", split)
        lbl_dir = os.path.join(root, "labels", split)

        image_files = [
            f
            for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        image_files = sorted(set(image_files))[:max_samples] if max_samples != None else sorted(set(image_files)) # Optionally limit number of samples for faster testing

        self.images = []
        self.masks = []
        for fname in image_files:
            img_path = os.path.join(img_dir, fname)
            base, _ = os.path.splitext(fname)
            lbl_name = base + ".png"
            lbl_path = os.path.join(lbl_dir, lbl_name)
            if os.path.exists(lbl_path):
                self.images.append(img_path)
                self.masks.append(lbl_path)

        if len(self.images) == 0:
            raise RuntimeError(f"No image/label pairs found in {img_dir} / {lbl_dir}")

        # This is used by load_raw_image in DataService / trainer_tools
        # so exposing .images is enough.
        self.image_transform = transforms.Compose(
            [
                transforms.Resize(
                    size=image_size,
                    interpolation=Image.BILINEAR
                ),
                transforms.ToTensor(),
            ]
        )
        self.mask_resize = transforms.Resize(
            size=image_size,
            interpolation=Image.NEAREST
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        """
            IMPORTANT: returns (item, uid, target) only.
                - item: transformed input (e.g. image tensor)
                - uid: unique identifier for the sample (e.g. filename)
                - target: transformed label/target (e.g. mask tensor)
                - metadata: optional dict with any additional info (e.g. original paths)
        """
        return self.get_items(idx, include_metadata=True, include_labels=True, include_images=True)

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):

        img_path = self.images[idx]
        mask_path = self.masks[idx]
        uid = os.path.basename(img_path)

        metadata = {
            'img_path': img_path,
            'mask_path': mask_path,
        } if include_metadata else None

        # Process images
        img_t = None
        if include_images:
            img = Image.open(img_path).convert("RGB")
            img_t = self.image_transform(img)

        # Process labels/masks
        # # # Sample wise segmentation
        # mask_t = None
        # if include_labels:
        #     mask = Image.open(mask_path)
        #     mask_r = self.mask_resize(mask)
        #     mask_np = np.array(mask_r, dtype=np.int64)
        #     mask_t = torch.from_numpy(mask_np) # [H, W] int64
        # return img_t, uid, mask_t, metadata
        # # Instance wise segmentaiton
        # Process labels/masks
        mask_t_instances = list()
        mask_t = None
        if include_labels:
            mask = Image.open(mask_path)
            mask_r = self.mask_resize(mask)
            mask_np = np.array(mask_r, dtype=np.int64)
            mask_t = torch.from_numpy(mask_np) # [H, W] int64

            # Format labels to register multiple instance_ids
            lbl_max = mask_t.max().item()
            for i in range(1, lbl_max + 1):
                m = torch.zeros_like(mask_t)
                m[mask_t == i] = i # Assign class ID as instance ID for simplicity; if set to 1, all instances of the same class would be merged...
                mask_t_instances.append(m)
        return img_t, uid, mask_t_instances, metadata

def seg_collate(batch):
    """Collate WL per-sample tuples for instance-segmentation.

    Each item is ``(img, uid, instances, metadata)`` where ``instances`` is a
    LIST of per-instance mask tensors ([H, W], pixel value = class id). The
    default torch collate cannot batch variable-length instance lists, so we
    keep them as a Python list (one entry per sample). Empty instances (all
    background) are filtered out so every kept instance is a real annotation.

    Returns:
        images: FloatTensor [B, C, H, W]
        ids: list[str] of length B
        labels: list[B] where labels[s] is a list of instance mask tensors
        metas: list[B] of metadata dicts
    """
    images = torch.stack([b[0] for b in batch], dim=0)
    ids = [b[1] for b in batch]
    labels = [
        [m for m in (b[2] or []) if torch.as_tensor(m).max() > 0] if isinstance(b[2], list) else b[2]
        for b in batch
    ]
    metas = [b[3] if len(b) > 3 else None for b in batch]
    return images, ids, labels, metas

## 3. The model (inlined `utils/model.py`)

A compact U-Net (`SmallUNet`) with `task_type="segmentation"` so WeightsLab renders masks in Weights Studio.

In [ ]:
# ===== utils/model.py =====
# =============================================================================
# Small UNet-ish segmentation model
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, image_size=256):
        super().__init__()
        # For WeightsLab
        self.task_type = "segmentation"
        self.num_classes = num_classes
        self.class_names = ["Background", "Ego Road", "Driveable Area", "Lane Line 1", "Lane Line 2", "Lane Line 3"]
        self.input_shape = (1, in_channels, image_size, image_size)

        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(64 + 64, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(32 + 32, 32)

        self.head = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        # Bottleneck
        b = self.bottleneck(p2)

        # Decoder
        u2 = self.up2(b)
        # Important: no `if` on shapes; always interpolate
        u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        logits = self.head(d1) # [B, C, H, W]
        return logits

## 4. Losses & metrics (inlined `utils/criterions.py`)

Per-sample and per-instance **Dice** (metric) and **BCE** (loss). `PerInstance*` returns one value per annotation (auto-saved at `(sample_id, annotation_id)`); `PerSample*` aggregates to one value per sample.

In [ ]:
# ===== utils/criterions.py =====
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# Per-instance / per-sample segmentation criterions (Dice + BCE)
# =============================================================================
# The segmentation dataset yields, per sample, a LIST of instance masks
# (each [H, W] with pixel value = class id). These criterions compute Dice and
# BCE for every instance against the model's per-class probability map, then:
# * PerInstance* returns a flat tensor (one value per instance, ordered
# sample-major) — wrapped with `per_instance=True` so WL auto-saves it at
# (sample_id, annotation_id).
# * PerSample* aggregates instances to one value per sample (mean) — wrapped
# with `per_sample=True` for the per-sample dashboards.
# The instance ordering matches the `batch_idx` passed by the training loop
# (built from the same per-sample instance lists), so WL maps each value to the
# correct annotation.

_EPS = 1e-6


def _instance_dice_bce(outputs, labels, **kwargs):
    """Compute per-instance Dice and BCE for a batch.

    Args:
        outputs: logits [B, C, H, W].
        labels: list[B]; labels[s] is a list of instance masks ([H, W], value = class id).

    Returns:
        (dice_per_sample, bce_per_sample) where each is a list[B] of 1-D tensors
        holding one value per instance for that sample (empty tensor if none).
        Values are kept on the outputs' device; BCE retains grad, Dice is a metric.
    """
    probs = torch.softmax(outputs, dim=1) # [B, C, H, W], differentiable
    B, C = probs.shape[0], probs.shape[1]
    device = outputs.device

    # Per-class weight vector [C] (optional). Applied as a SCALAR multiplier on
    # each instance's BCE, keyed by that instance's class — NOT as the per-pixel
    # `weight=` arg of F.binary_cross_entropy (that expects a tensor broadcastable
    # to [H, W], so a [C] class vector would raise a shape error).
    weights = kwargs.get("weights", None)
    if weights is not None:
        weights = torch.as_tensor(weights, device=device, dtype=probs.dtype)

    dice_per_sample, bce_per_sample = [], []
    for s in range(B):
        insts = labels[s] if s < len(labels) else []
        # Accept a stacked [N, H, W] tensor or a list of [H, W] masks.
        if isinstance(insts, torch.Tensor):
            insts = [insts[i] for i in range(insts.shape[0])] if insts.ndim == 3 else [insts]

        dices, bces = [], []
        for m in insts:
            m = torch.as_tensor(m, device=device)
            cls = int(m.max().item())
            ch = cls if 0 <= cls < C else 0
            gt = (m > 0).float()
            p = probs[s, ch].clamp(_EPS, 1.0 - _EPS) # [H, W]
            inter = (p * gt).sum()
            dice = (2.0 * inter + _EPS) / (p.sum() + gt.sum() + _EPS)
            bce = F.binary_cross_entropy(p, gt)
            if weights is not None:
                bce = bce * weights[ch] # scalar class weight for this instance
            dices.append(dice)
            bces.append(bce)

        dice_per_sample.append(torch.stack(dices) if dices else torch.zeros(0, device=device))
        bce_per_sample.append(torch.stack(bces) if bces else torch.zeros(0, device=device))

    return dice_per_sample, bce_per_sample


class PerSampleDice(nn.Module):
    """Mean Dice over a sample's instances → one value per sample ([B])."""

    def forward(self, outputs, labels):
        dice_per_sample, _ = _instance_dice_bce(outputs, labels)
        out = [d.mean() if d.numel() > 0 else torch.zeros((), device=outputs.device)
               for d in dice_per_sample]
        return torch.stack(out).detach()


class PerInstanceDice(nn.Module):
    """Dice per instance → flat tensor [total_instances] (sample-major order)."""

    def forward(self, outputs, labels):
        dice_per_sample, _ = _instance_dice_bce(outputs, labels)
        flat = [v for d in dice_per_sample for v in d] if any(d.numel() for d in dice_per_sample) else []
        if not flat:
            return torch.zeros(0, device=outputs.device)
        return torch.stack(flat).detach()


class PerSampleBCE(nn.Module):
    """Mean BCE over a sample's instances → one value per sample ([B])."""
    def __init__(self, weights=None):
        super().__init__()
        self.register_buffer("weights", torch.tensor(weights) if weights is not None else None)

    def forward(self, outputs, labels):
        _, bce_per_sample = _instance_dice_bce(outputs, labels, weights=self.weights)
        out = [b.mean() if b.numel() > 0 else torch.zeros((), device=outputs.device)
               for b in bce_per_sample]
        return torch.stack(out)


class PerInstanceBCE(nn.Module):
    """BCE per instance → flat tensor [total_instances] (sample-major order)."""
    def __init__(self, weights=None):
        super().__init__()
        self.register_buffer("weights", torch.tensor(weights) if weights is not None else None)

    def forward(self, outputs, labels):
        _, bce_per_sample = _instance_dice_bce(outputs, labels, weights=self.weights)
        flat = [v for b in bce_per_sample for v in b] if any(b.numel() for b in bce_per_sample) else []
        if not flat:
            return torch.zeros(0, device=outputs.device)
        return torch.stack(flat).detach()

## 5. Configuration

Every tunable lives here in one dict - this is `config.yaml` inlined, with its comments. Wrapping it with `flag="hyperparameters"` lets Weights Studio read (and live-edit) these values while training. `data_root` points at the sample fetched above.

In [ ]:
config = {
    # -- Global configuration ------------------------------------------------
    "device": "auto",                         # 'auto' -> resolved to `device` from the Imports cell
    "experiment_name": "seg_bdd_training",    # name shown in Weights Studio
    "training_steps_to_do": 500,               # finite cap for this demo; set to None to train until you interrupt
    "root_log_dir": None,                       # None -> a temp dir is created (history + dataframe land here)

    "checkpoint_manager": {
        "load_config": False,                   # don't reload a previous checkpoint's config, so edits here take effect
    },

    # Initially compute natural sorting values, e.g. average intensity (see the example README).
    "compute_natural_sort": True,

    # -- Experiment parameters ----------------------------------------------
    "eval_full_to_train_steps_ratio": 5,        # run a full eval every N steps
    "experiment_dump_to_train_steps_ratio": 5,
    "write_export_ratio": 100,                  # export history + dataframe every N steps (main.py default)
    "tqdm_display": True,
    "is_training": False,                       # start training immediately or not

    # -- Serving (the notebook serves via gRPC + a bore tunnel; see the Serve cell) --
    "serving_grpc": True,
    "serving_cli": True,

    # -- Global dataframe storage -------------------------------------------
    "ledger_enable_h5_persistence": True,
    "ledger_enable_flushing_threads": True,
    "ledger_flush_max_rows": 100,
    "ledger_flush_interval": 60.0,

    # -- Data ----------------------------------------------------------------
    "num_classes": 6,
    "class_names": ["Background", "Ego Road", "Driveable Area", "Lane Line 1", "Lane Line 2", "Lane Line 3"],
    "ignore_index": 255,                        # void pixels (main.py default)
    "image_size": 128,                          # images/masks resized to image_size
    "data_root": DATA_ROOT,                     # folder extracted from your Drive zip (see the data-fetch cell)
    "data": {
        "train_loader": {"batch_size": 2, "shuffle": True, "max_samples": 256},
        "test_loader": {"batch_size": 2, "shuffle": False, "max_samples": 100},
    },

    # -- Optimizer -----------------------------------------------------------
    "optimizer": {"lr": 1e-3},                  # Adam learning rate (main.py default)
}

# Resolve the 'auto' device to the one picked in the Imports cell.
if config.get("device", "auto") == "auto":
    config["device"] = str(device)

# Register the config so Weights Studio can read (and live-edit) it while training.
wl.watch_or_edit(config, flag="hyperparameters", name=config["experiment_name"],
                 defaults=config, poll_interval=1.0)

# Logging directory: None -> a temp dir is created.
if not config.get("root_log_dir"):
    config["root_log_dir"] = tempfile.mkdtemp(prefix="weightslab_seg_")
os.makedirs(config["root_log_dir"], exist_ok=True)
log_dir = config["root_log_dir"]

# Schedule knobs used by the training loop.
eval_full_to_train_steps_ratio = config["eval_full_to_train_steps_ratio"]
write_export_ratio = config.get("write_export_ratio", 100)
tqdm_display = config.get("tqdm_display", True)
print("Experiment logs ->", log_dir)

## 6. Build data, model & optimizer

The heart of WeightsLab: each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave like the originals but report their state and per-sample signals. The loaders use `collate_fn=seg_collate` to preserve per-instance masks. (Same wiring as the example's `main.py`.)

In [ ]:
# --- Hyperparameters read back after watch_or_edit ---
num_classes = int(config["num_classes"])
class_names = config["class_names"]
ignore_index = int(config["ignore_index"])
image_size = int(config["image_size"])

# --- Data (from the Drive zip fetched above) ---
data_root = config.get("data_root", DATA_ROOT)
if os.path.exists(data_root):
    print(f"Using data root: {data_root}")
else:
    raise FileNotFoundError(
        f"Data root not found: {data_root!r}. Run the data-fetch cell above first.")

train_cfg = config.get("data", {}).get("train_loader", {})
test_cfg = config.get("data", {}).get("test_loader", {})

_train_dataset = BDD100kSegDataset(
    root=data_root,
    split="train",
    num_classes=num_classes,
    class_names=class_names,
    ignore_index=ignore_index,
    image_size=image_size,
    max_samples=train_cfg.get("max_samples", None),
)
_val_dataset = BDD100kSegDataset(
    root=data_root,
    split="val",
    num_classes=num_classes,
    class_names=class_names,
    ignore_index=ignore_index,
    image_size=image_size,
    max_samples=test_cfg.get("max_samples", None),
)

train_loader = wl.watch_or_edit(
    _train_dataset, flag="data", loader_name="train_loader",
    batch_size=train_cfg.get("batch_size", 2), shuffle=train_cfg.get("shuffle", True),
    compute_hash=False, is_training=True,
    array_autoload_arrays=False, array_return_proxies=True, array_use_cache=True,
    preload_labels=False, collate_fn=seg_collate,
)
test_loader = wl.watch_or_edit(
    _val_dataset, flag="data", loader_name="test_loader",
    batch_size=test_cfg.get("batch_size", 2), shuffle=test_cfg.get("shuffle", False),
    compute_hash=False, is_training=False,
    array_autoload_arrays=False, array_return_proxies=True, array_use_cache=True,
    preload_labels=True, collate_fn=seg_collate,
)

_model = SmallUNet(in_channels=3, num_classes=num_classes, image_size=image_size).to(device)
model = wl.watch_or_edit(_model, flag="model", device=device, compute_dependencies=True)

lr = config.get("optimizer", {}).get("lr", 1e-3)
_optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = wl.watch_or_edit(_optimizer, flag="optimizer")

## 7. Train & eval steps (+ tracked signals)

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `_run_instance_signals` computes and logs the per-sample **and** per-instance Dice + BCE, and `wl.save_signals(...)` records custom per-sample diagnostics. Class weights are computed from the training split, then the tracked signals are built - exactly as in `main.py`.

In [ ]:
# ===== main.py: train / eval step functions =====
def _instance_batch_idx(labels):
    """Flat instance→sample map (sample-major) matching the PerInstance* ordering."""
    return torch.tensor(
        [s for s, insts in enumerate(labels) for _ in insts],
        dtype=torch.long,
    )


def _run_instance_signals(sig, outputs, labels, ids, preds, return_metric=False):
    """Compute + log/save the per-sample AND per-instance Dice (metric) and BCE (loss)."""
    bce_sample = sig["bce_sample"](outputs, labels, batch_ids=ids, preds=preds)
    dice_sample = sig["dice_sample"](outputs, labels, batch_ids=ids) # Register processed predictions one time only

    sig["dice_instance"](outputs, labels, batch_ids=ids) # Register processed predictions one time only
    sig["bce_instance"](outputs, labels, batch_ids=ids)

    avg_loss = 0.5 * dice_sample + 0.5 * bce_sample
    wl.save_signals({"combined_bce_dice_per_sample": avg_loss}, ids) # Save the per-sample aggregate loss for backward step
    if return_metric:
        return avg_loss, dice_sample
    return avg_loss


def _user_custom_signals(preds, labels):
    """Example of user-defined custom signals to save additional info to WL."""
    return {
        "preds_classes_per_sample": [
            preds[i].unique() for i in range(preds.shape[0])
        ],
        "target_classes_per_sample": [
            torch.unique(torch.cat(labels[i])) for i in range(len(labels))
        ],
        "tp_classes_per_sample": [
            torch.tensor([c for c in torch.unique(torch.cat(labels[i])) if c in preds[i].unique()])
            for i in range(len(labels))
        ],
        "fp_classes_per_sample": [
            torch.tensor([c for c in preds[i].unique() if c not in torch.unique(torch.cat(labels[i]))])
            for i in range(len(labels))
        ],
        "fn_classes_per_sample": [
            torch.tensor([c for c in torch.unique(torch.cat(labels[i])) if c not in preds[i].unique()])
            for i in range(len(labels))
        ],
    }


def train(loader, model, optimizer, sig, device):
    """
    Single training step using the tracked dataloader + watched loss.

    loader yields (inputs, ids, labels, metadata) because of DataSampleTrackingWrapper.
    `labels` is per sample a LIST of instance masks (see utils/data.seg_collate).
    """
    with guard_training_context:
        (inputs, ids, labels, _) = next(loader)
        inputs = inputs.to(device)
        labels = [[m.to(device) for m in insts] for insts in labels] # per-sample list of instances

        optimizer.zero_grad()
        outputs = model(inputs) # [B,C,H,W]
        preds = outputs.argmax(dim=1) # [B,H,W]

        # Per-instance + per-sample Dice/BCE (tracked & saved at annotation level).
        loss_per_sample = _run_instance_signals(sig, outputs, labels, ids, preds=preds)

        # Backward loss: per-sample CrossEntropy over the merged semantic mask.
        loss = loss_per_sample.mean()

        loss.backward()
        optimizer.step()

        # I want to see in the UI the per-sample classes predicted by the model and what classes are missing compared to the target (for error analysis)
        wl.save_signals(
            _user_custom_signals(preds, labels),
            ids
        ) # Save the per-sample predictions for visualization

    return float(loss.detach().cpu().item())


def test(loader, model, sig, device, test_loader_len):
    """Full evaluation pass over the val loader."""
    losses = 0.0
    dices = 0.0
    with guard_testing_context, torch.no_grad():
        for inputs, ids, labels, _ in loader:
            inputs = inputs.to(device)
            labels = [[m.to(device) for m in insts] for insts in labels] # per-sample list of instances

            outputs = model(inputs)
            preds = outputs.argmax(dim=1) # [B,H,W]

            # Per-instance + per-sample Dice/BCE (tracked & saved at annotation level).
            loss_per_sample, dice_sample = _run_instance_signals(sig, outputs, labels, ids, preds=preds, return_metric=True)

            losses += torch.mean(loss_per_sample) # Average over the batch and accumulate
            dices += torch.mean(dice_sample) # Average over the batch and accumulate

            # I want to see in the UI the per-sample classes predicted by the model
            wl.save_signals(_user_custom_signals(preds, labels), ids) # Save the per-sample predictions for visualization

    loss = float((losses / test_loader_len).detach().cpu().item())
    dice = float((dices / test_loader_len).detach().cpu().item())
    return loss, dice * 100.0 # Return average Dice as percentage


# ===== main.py: tracked per-sample / per-instance Dice (metric) + BCE (loss) signals =====
# --- Per-instance + per-sample Dice (metric) and BCE (loss) signals ---
# Each instance mask is scored against the model's per-class probability map.
# per_instance=True auto-saves one value per (sample_id, annotation_id);
# per_sample=True logs the per-sample aggregate (mean over the sample's instances).
def _make_seg_signals(split: str, weights: dict = None) -> dict:
    return {
        "dice_sample": wl.watch_or_edit(
            PerSampleDice(), flag="metric",
            name=f"{split}_dice/sample", per_sample=True, log=True,
        ),
        "dice_instance": wl.watch_or_edit(
            PerInstanceDice(), flag="metric",
            name=f"{split}_dice/instance", per_instance=True, log=True,
        ),
        "bce_sample": wl.watch_or_edit(
            PerSampleBCE(weights=weights), flag="loss",
            name=f"{split}_bce/sample", per_sample=True, log=True,
        ),
        "bce_instance": wl.watch_or_edit(
            PerInstanceBCE(weights=weights), flag="loss",
            name=f"{split}_bce/instance", per_instance=True, log=True,
        ),
    }


def compute_class_weights(dataset, num_classes, ignore_index=255, max_samples=100):
    print("\n" + "=" * 60, flush=True)
    print(f"Computing class weights for {num_classes} classes (max {max_samples} samples)...", flush=True)
    class_counts = np.zeros(num_classes, dtype=np.float64)
    num_samples = min(len(dataset), max_samples)

    for idx in tqdm.tqdm(range(num_samples), desc=" Analyzing Distribution"):
        _, _, label, _ = dataset.get_items(idx, include_labels=True) # Get the label/mask for this sample
        label_np = label.numpy() if hasattr(label, 'numpy') else np.array(label)
        for c in range(num_classes):
            class_counts[c] += (label_np == c).sum()

    class_counts = np.maximum(class_counts, 1) # Avoid div by zero
    total_pixels = class_counts.sum()
    class_weights = total_pixels / (num_classes * class_counts)
    class_weights = class_weights / class_weights.mean() # Normalize

    print("\nClass distribution and weights:", flush=True)
    for c in range(num_classes):
        pct = (class_counts[c] / total_pixels) * 100
        print(f"Class {c}: {pct:6.2f}% -> weight: {class_weights[c]:.3f}", flush=True)
    print("=" * 60 + "\n", flush=True)
    return torch.FloatTensor(class_weights).to(device)

weights = compute_class_weights(_train_dataset, num_classes)

train_sig = _make_seg_signals("train", weights=weights)
test_sig = _make_seg_signals("test", weights=weights)

## 8. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server and opens a public `bore.pub:<port>` tunnel (printed below) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run a finite loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [ ]:
wl.serve(serving_grpc=config.get("serving_grpc", True), serving_bore=True)

wl.start_training(timeout=3)

# training_steps_to_do may be None (open-ended); cap it here so the notebook run is finite.
max_steps = config["training_steps_to_do"] or 500
train_range = tqdm.tqdm(range(max_steps), desc="Training") if tqdm_display else range(max_steps)
test_loss, test_metric = None, None
start_time = time.time()
for train_step in train_range:
    age = model.get_age() if hasattr(model, "get_age") else train_step

    # Train one step
    train_loss = train(train_loader, model, optimizer, train_sig, device)

    # Full evaluation pass
    if age == 0 or age % eval_full_to_train_steps_ratio == 0:
        test_loader_len = len(test_loader)
        test_loader_it = tqdm.tqdm(test_loader, desc="Evaluating", leave=False) if tqdm_display else test_loader
        test_loss, test_metric = test(test_loader_it, model, test_sig, device, test_loader_len)

    # Periodic history + dataframe export (JSON/CSV snapshots to root_log_dir)
    if age > 0 and age % write_export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    if tqdm_display:
        train_range.set_description("Step")
        train_range.set_postfix(
            train_loss=f"{train_loss:.4f}",
            test_loss=f"{test_loss:.4f}" if test_loss is not None else "N/A",
            dice=f"{test_metric:.2f}%" if test_metric is not None else "N/A",
        )

print("\n" + "=" * 60)
print(f" Training completed in {time.time() - start_time:.2f} seconds")
print(f" Logs saved to: {log_dir}")
print("=" * 60)

# Final export of signal history and data grid to root_log_dir
wl.write_history()
wl.write_dataframe()

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this notebook's
backend using the `bore.pub:<port>` printed in the Expose section:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 - opens http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 - the host:port from the Expose section
```

Then open **http://localhost:5173**. Prefer local-only? Run this example directly on your machine
(`weightslab start example` selects a bundled example) and launch the UI next to it - no tunnel.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>